# 🖼️ AI Image Captioner — Multi-Model, Multi-Source

**Author:** Mitadru Deb  
**Status:** Beginner Portfolio Project (LLM / Multimodal AI Engineering)  
**Last Updated:** November 2026

---

## What This Project Does

This project is an **end-to-end AI image captioning application** capable of:

1. **Direct Upload:** Upload any image from your computer to generate a descriptive caption.
2. **Web Scraping:** Enter any webpage URL (e.g. Wikipedia) — the app automatically discovers, filters, downloads, and captions all images on that page.
3. **Multi-Model Comparison:** Easily toggle between two popular open-source vision-language models:
   - **BLIP** (~0.25B parameters from Salesforce)
   - **Florence-2** (~0.23B parameters from Microsoft)

### Models Comparison

| Model | Params | Strengths | Architecture |
|-------|--------|-----------|--------------|
| [Salesforce/blip-image-captioning-base](https://huggingface.co/Salesforce/blip-image-captioning-base) | ~0.25B | Fast, concise captions, lightweight | ViT-B/16 + BERT text decoder |
| [microsoft/Florence-2-base](https://huggingface.co/microsoft/Florence-2-base) | ~0.23B | Rich, detailed scene descriptions, prompt-guided | DaViT vision encoder + sequence-to-sequence LM |

> 💡 **Hardware Compatibility:** Both models are small enough to run smoothly on standard consumer CPUs, yet automatically utilize CUDA GPUs if available for accelerated inference. Works identically across **Windows, Linux, macOS, and Google Colab**.

---

## Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Import Libraries & Environment Check](#2-import-libraries--environment-check)
3. [Load AI Models (BLIP & Florence-2)](#3-load-ai-models-blip--florence-2)
4. [Core Pipeline Functions](#4-core-pipeline-functions)
   - 4a. Caption a Single Image
   - 4b. Web Scraping & URL Image Captioning
5. [Interactive Gradio Web UI](#5-interactive-gradio-web-ui)
6. [What I Learned & Architecture Notes](#6-what-i-learned--architecture-notes)

---

## 1. Setup & Installation

Run the cell below to install the required libraries.  
- In **Google Colab**, running this cell installs all dependencies into your session.
- If running **locally**, you can also run `pip install -r requirements.txt` in your terminal.

> ⚠️ **Note on Dependencies:** We pin `transformers>=4.41.2,<5.0.0` for full compatibility with Florence-2, and avoid heavyweight CUDA compiler dependencies like `flash_attn` by using PyTorch's native Scaled Dot-Product Attention (SDPA).

In [ ]:
# Install required dependencies (run once)
!pip install -q "transformers>=4.41.2,<5.0.0" torch torchvision pillow gradio requests beautifulsoup4 einops timm


---

## 2. Import Libraries & Environment Check

Here we import all core modules and inspect the runtime environment:
- **`torch` & `torchvision`**: Deep learning framework and tensor operations
- **`transformers`**: Hugging Face pipeline for model weights, tokenizers, and processors
- **`PIL` (Pillow)**: Image decoding, format normalization (RGB), and resizing
- **`requests` & `bs4` (BeautifulSoup)**: HTTP fetching and HTML parsing for web scraping
- **`gradio`**: Web UI with responsive components and instant sharing links

In [ ]:
import os
import platform
import requests
import numpy as np
from io import BytesIO
from PIL import Image
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from unittest.mock import patch

import torch
from transformers.dynamic_module_utils import get_imports
from transformers import (
    AutoProcessor,
    BlipForConditionalGeneration,   # for Salesforce BLIP
    AutoModelForCausalLM,            # for Microsoft Florence-2
)
import gradio as gr

# Verify runtime environment
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:       {torch.cuda.get_device_name(0)}")
else:
    print("Running on:       CPU (Standard)")

---

## 3. Load AI Models (BLIP & Florence-2)

Both models are downloaded from Hugging Face Hub and cached locally.  

### How the Vision-Language Pipeline Works

```
┌──────────────┐     ┌───────────────────────┐     ┌────────────────────┐     ┌──────────────┐
│ Raw Image    │ ──► │ AutoProcessor         │ ──► │ Transformer Model  │ ──► │ Text Caption │
│ (RGB Bitmap) │     │ (Normalize + Tokens)  │     │ (Generates Tokens) │     │ (Decoded)    │
└──────────────┘     └───────────────────────┘     └────────────────────┘     └──────────────┘
```

### Engineering Detail: Cross-Platform Florence-2 Loading
Florence-2's remote configuration references `flash_attn`. On systems without compilation tools (like Windows or CPU Colab), we intercept the import check using `unittest.mock.patch` to seamlessly fall back to PyTorch's native Scaled Dot-Product Attention (`sdpa`).

In [ ]:
# --- Device & Precision Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Target device: {device}")
print(f"Target dtype:  {torch_dtype}")
print()

# =============================================
# Model 1: BLIP (Salesforce) — ~0.25B params
# =============================================
print("Loading BLIP processor & model...")
blip_processor = AutoProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch_dtype,
).to(device)
print("✅ BLIP loaded successfully!")

# =============================================
# Model 2: Florence-2 (Microsoft) — ~0.23B params
# =============================================
print("Loading Florence-2 processor & model...")

# Dynamic patch to ensure compatibility without flash_attn
def workaround_fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

florence_processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-base",
    trust_remote_code=True,
)

with patch("transformers.dynamic_module_utils.get_imports", workaround_fixed_get_imports):
    florence_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-base",
        torch_dtype=torch_dtype,
        trust_remote_code=True,
    ).to(device)

print("✅ Florence-2 loaded successfully!")
print("\n🎉 Both models are in memory and ready for inference!")

---

## 4. Core Pipeline Functions

We encapsulate the captioning logic into two clean, reusable functions:

| Function | Input | Purpose |
|----------|-------|---------|
| `generate_caption()` | PIL Image + Model Name | Runs token generation and decoding on an image |
| `caption_image()` | Gradio NumPy array + Model Name | Handles user-uploaded images from the UI |
| `caption_from_url()` | Web URL + Model Name + Max Images | Scrapes HTML, filters invalid images, and captions each valid image |

### 4a. Caption a Single Image

This function prepares the tensors, applies the model-specific prompt, and decodes the resulting token sequence back into natural language.

In [ ]:
def generate_caption(pil_image: Image.Image, model_name: str) -> str:
    """
    Generate a caption for a single PIL image using either BLIP or Florence-2.

    Args:
        pil_image:  PIL.Image in RGB mode.
        model_name: Selected model ("BLIP" or "Florence-2").

    Returns:
        String containing the generated description.
    """
    if model_name == "BLIP":
        # BLIP text prompt guiding generation
        inputs = blip_processor(
            images=pil_image,
            text="the image of",
            return_tensors="pt",
        ).to(device)
        if device == "cuda":
            inputs["pixel_values"] = inputs["pixel_values"].to(torch_dtype)

        output_ids = blip_model.generate(**inputs, max_new_tokens=50)
        caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)

    elif model_name == "Florence-2":
        # Florence-2 uses task tokens (<CAPTION>, <DETAILED_CAPTION>, <MORE_DETAILED_CAPTION>)
        task_prompt = "<DETAILED_CAPTION>"
        inputs = florence_processor(
            text=task_prompt,
            images=pil_image,
            return_tensors="pt",
        ).to(device)
        if device == "cuda":
            inputs["pixel_values"] = inputs["pixel_values"].to(torch_dtype)

        output_ids = florence_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3,
            do_sample=False,
        )

        raw_text = florence_processor.batch_decode(
            output_ids, skip_special_tokens=False
        )[0]
        parsed = florence_processor.post_process_generation(
            raw_text,
            task=task_prompt,
            image_size=(pil_image.width, pil_image.height),
        )
        caption = parsed.get(task_prompt, raw_text)

    else:
        caption = f"Error: Unsupported model '{model_name}'."

    return caption


def caption_image(input_image: np.ndarray, model_name: str) -> str:
    """
    Gradio callback for the 'Upload Image' tab.
    """
    if input_image is None:
        return "⚠️ Please upload an image first."

    pil_image = Image.fromarray(input_image).convert("RGB")
    return generate_caption(pil_image, model_name)


print("✅ caption_image() defined.")

### 4b. Web Scraping & URL Image Captioning

This function:
1. Performs an HTTP GET request with realistic user-agent headers.
2. Extracts all `<img>` tags from the HTML via BeautifulSoup.
3. Filters out SVGs (vector art), data URLs, and tiny thumbnails (< 400px area).
4. Resolves relative image paths into absolute URLs using `urllib.parse.urljoin`.
5. Downloads and generates captions for up to `max_images`.

In [ ]:
def caption_from_url(url: str, model_name: str, max_images: int = 5) -> str:
    """
    Scrape images from a webpage URL and caption each valid image.

    Args:
        url:         Webpage URL (e.g. https://en.wikipedia.org/wiki/Computer)
        model_name:  Selected model ("BLIP" or "Florence-2")
        max_images:  Upper limit on images to process (default: 5)

    Returns:
        Formatted summary string with all image links and captions.
    """
    if not url or not url.strip():
        return "⚠️ Please enter a valid URL."

    url = url.strip()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Step 1: Fetch HTML
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        return f"❌ Could not fetch webpage: {e}"

    # Step 2: Parse images
    soup = BeautifulSoup(response.text, "html.parser")
    img_elements = soup.find_all("img")

    if not img_elements:
        return "No images found on this page."

    # Step 3: Iterate, filter, download, and caption
    results = []
    captioned_count = 0

    for idx, img_element in enumerate(img_elements, start=1):
        if captioned_count >= max_images:
            break

        # Handle src, data-src, or srcset
        img_url = img_element.get("src") or img_element.get("data-src")
        if not img_url and img_element.has_attr("srcset"):
            img_url = img_element["srcset"].split()[0]
        if not img_url:
            continue

        # Skip vector graphics and data URIs
        if ".svg" in img_url.lower() or img_url.startswith("data:"):
            continue

        # Resolve relative URLs
        img_url = urljoin(url, img_url)
        if not img_url.startswith("http"):
            continue

        # Download image and run captioner
        try:
            img_response = requests.get(img_url, timeout=10, headers=headers)
            pil_image = Image.open(BytesIO(img_response.content))

            # Filter out tiny icon / tracking pixels
            width, height = pil_image.size
            if width * height < 400 or width < 30 or height < 30:
                continue

            pil_image = pil_image.convert("RGB")
            caption = generate_caption(pil_image, model_name)

            captioned_count += 1
            results.append(
                f"📷 Image {captioned_count}:\n"
                f"   URL: {img_url}\n"
                f"   Caption: {caption}\n"
            )

        except (OSError, Exception):
            continue

    if not results:
        return "No captionable images found (images may be SVGs, too small, or blocked)."

    header = (
        f"✅ Captioned {captioned_count} image(s) from: {url}\n"
        f"Model: {model_name}\n"
        f"{'=' * 60}\n\n"
    )
    return header + "\n".join(results)


print("✅ caption_from_url() defined.")

---

## 5. Interactive Gradio Web UI

We construct a tabbed interface using **Gradio Blocks**:

- **Tab 1: 📤 Upload Image:** Drag and drop any image, choose your model, and get immediate text descriptions.
- **Tab 2: 🌐 Caption from URL:** Paste any webpage URL, pick the model and image count, and see batch captions.

> 🌐 **Google Colab Note:** When run in Google Colab, Gradio automatically creates a public temporary URL (`.gradio.live`) so you can share and test your app anywhere!

In [ ]:
# Detect if executing in Google Colab environment
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

model_choices = ["BLIP", "Florence-2"]

with gr.Blocks(title="AI Image Captioner") as app:

    gr.Markdown(
        """
        # 🖼️ AI Image Captioner
        Generate rich image descriptions using open-source Vision-Language Models.  
        **Upload an image** or **paste a URL** to describe images from any webpage.

        Models: **BLIP** (Salesforce, ~0.25B) · **Florence-2** (Microsoft, ~0.23B)
        """
    )

    with gr.Tabs():

        # ---- Tab 1: Upload Image ----
        with gr.TabItem("📤 Upload Image"):
            with gr.Row():
                with gr.Column():
                    upload_image = gr.Image(
                        type="numpy",
                        label="Upload an Image",
                    )
                    upload_model = gr.Dropdown(
                        choices=model_choices,
                        value="BLIP",
                        label="Select AI Model",
                    )
                    upload_btn = gr.Button("Generate Caption", variant="primary")

                with gr.Column():
                    upload_output = gr.Textbox(
                        label="Generated Caption",
                        lines=5,
                    )

            upload_btn.click(
                fn=caption_image,
                inputs=[upload_image, upload_model],
                outputs=upload_output,
            )

        # ---- Tab 2: Caption from URL ----
        with gr.TabItem("🌐 Caption from URL"):
            with gr.Row():
                with gr.Column():
                    url_input = gr.Textbox(
                        label="Webpage URL",
                        placeholder="https://en.wikipedia.org/wiki/Computer",
                        lines=1,
                    )
                    url_model = gr.Dropdown(
                        choices=model_choices,
                        value="BLIP",
                        label="Select AI Model",
                    )
                    url_max = gr.Slider(
                        minimum=1,
                        maximum=15,
                        value=5,
                        step=1,
                        label="Max Images to Caption",
                    )
                    url_btn = gr.Button("Scrape & Caption", variant="primary")

                with gr.Column():
                    url_output = gr.Textbox(
                        label="Captions",
                        lines=15,
                    )

            url_btn.click(
                fn=caption_from_url,
                inputs=[url_input, url_model, url_max],
                outputs=url_output,
            )

    gr.Markdown(
        """
        ---
        *Built with [Hugging Face Transformers](https://huggingface.co/docs/transformers) & [Gradio](https://gradio.app) · 100% open-source, zero API cost.*
        """
    )

# Launch the app
# In Google Colab, share=True creates a public shareable URL
app.launch(share=IS_COLAB)

---

## 6. What I Learned & Architecture Notes

### Key Takeaways for LLM Engineering Interviews

1. **Vision-Language Model Fundamentals:** Multimodal models align two distinct embedding spaces (vision tokens from a ViT/DaViT encoder and text tokens from a language model decoder). Both BLIP and Florence-2 illustrate effective cross-attention mechanisms at lightweight scale (~0.25B parameters).

2. **Open-Source Model Efficiency:** Models under 300M parameters can run inference comfortably on consumer CPU hardware with low latency, proving that massive frontier models are not always required for focused vision-to-text tasks.

3. **Real-World Data Ingestion:** Web scraping with `BeautifulSoup` requires robust handling of edge cases (relative vs absolute URLs with `urljoin`, SVG/vector filtering, tiny tracking pixel exclusion, and network timeouts).

4. **Cross-Platform Engineering:** Handling dynamic imports (`flash_attn` fallback to SDPA) and maintaining proper tensor dtypes (`input_ids` as `torch.long`, `pixel_values` as float) are essential for production reliability across Windows, Linux, and Colab.

5. **Rapid UI Prototyping:** Gradio enables modular development with built-in queueing, tabbed navigation, and public tunneling for live portfolio demonstrations.